In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.chdir('/content/drive/My Drive/DL/final_project')
path = os.getcwd()
print('path: ' + path)

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

In [ ]:
!pip install -r Depth-Anything-V2/requirements.txt

In [ ]:
import cv2
import torch
import os
import pandas as pd
import numpy as np
import math
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
os.chdir("/content/drive/My Drive/DL/final_project/Depth-Anything-V2")
from depth_anything_v2.dpt import DepthAnythingV2
from depth_anything_v2.util.transform import Resize, NormalizeImage, PrepareForNet
os.chdir("/content/drive/My Drive/DL/final_project")


In [4]:
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
print(f"device: {DEVICE}")

In [6]:
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vitl'

In [ ]:
model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'Depth-Anything-V2/checkpoints/depth_anything_v2_vitl.pth', map_location=DEVICE))

In [ ]:
preprocess, postprocess, waternet = torch.hub.load('tnwei/waternet', 'waternet')

waternet.eval()

In [60]:
def custom_loss(disparity_map, target_coords, gt_depth, ref_coords, ref_distance, eps=1e-6):

    import matplotlib.pyplot as plt
    plt.imshow(disparity_map[0].detach().numpy(), cmap='plasma')
    plt.scatter(target_coords[0].detach().numpy(), target_coords[1].detach().numpy(), color='red', s=10)
    plt.show()

    disparity_value_ref = disparity_map[0][ref_coords[1], ref_coords[0]]
    disparity_value_target = disparity_map[0][target_coords[1], target_coords[0]]

    relative_depth_ref = 1.0 / (disparity_value_ref + eps)
    relative_depth_target = 1.0 / (disparity_value_target + eps)

    scale = ref_distance / relative_depth_ref

    predicted_depth = relative_depth_target * scale

    predicted_depth = torch.clamp(predicted_depth, max=10000)

    print(f'predicted depth: {predicted_depth.item():2f}, ground depth: {gt_depth.item():2f}')
    print(f'disparity_value_target: {disparity_value_target.item():2f}, disparity_value_ref: {disparity_value_ref.item():2f}')
    print(f'relative_depth_target: {relative_depth_target.item():2f}, relative_depth_ref: {relative_depth_ref.item():2f}')
    print(f'scale: {scale.item():2f}')

    loss = torch.mean((torch.log(predicted_depth + 1e-6) - torch.log(gt_depth + 1e-6)) ** 2)
    return loss

In [45]:
def prepare_data():
  file_path = 'length_measurements_reflected.csv'
  df = pd.read_csv(file_path)

  # Columns to keep
  columns_to_keep = ['Filename', 'pixel_location', 'Act. Range', 'ref_pixel', 'ref_distance']
  df = df[columns_to_keep]

  df.rename(columns={'Filename': 'VideoName'}, inplace=True)
  df.index = df.index + 2

  df = df.dropna(subset=['pixel_location'])

  df[['pixel_x', 'pixel_y']] = df['pixel_location'].str.extract(r'\((\d+),\s*(\d+)\)').astype(int)
  df[['ref_pixel_x', 'ref_pixel_y']] = df['ref_pixel'].str.extract(r'\((\d+),\s*(\d+)\)').astype(int)
  df['images'] = df.apply(lambda row: f'{row["VideoName"][:-4]}_number_{row.name}.jpg', axis=1)
  df['Act. Range'] = df['Act. Range'].astype(float)

  df = df.drop(columns=['VideoName', 'ref_pixel', 'pixel_location'])

  return df

In [ ]:
#print images with point on coordinate

df = prepare_data()
for index, row in df.iterrows():
    image_path = f'data_images/{row["images"]}'
    img = plt.imread(image_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.scatter(row['pixel_x'], row['pixel_y'], color='red', s=10)
    plt.title(f'Image {row["images"]}, Range: {row["Act. Range"]}')
    plt.axis('off')
    plt.show()

In [46]:
from torchvision.transforms import Compose

USE_WATERNET = True

def image2tensor(raw_image):
  if USE_WATERNET:
    return image2tensor_with_waternet(raw_image)
  else:
    return image2tensor_without_water_net(raw_image)

In [47]:
def image2tensor_with_waternet(raw_image, input_size=518):
        transform_resize = Compose([
            Resize(
                width=input_size,
                height=input_size,
                resize_target=False,
                keep_aspect_ratio=True,
                ensure_multiple_of=14,
                resize_method='lower_bound',
                image_interpolation_method=cv2.INTER_CUBIC,
            ),
        ])

        h, w = raw_image.shape[:2]

        image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)

        image = transform_resize({'image': image})['image']
        rgb_ten, wb_ten, he_ten, gc_ten = preprocess(image)
        image = waternet(rgb_ten, wb_ten, he_ten, gc_ten)
        image = postprocess(image)[0] / 255.0


        transform_prepare = Compose([
            NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            PrepareForNet(),
        ])

        image = transform_prepare({'image': image})['image']

        image = torch.from_numpy(image)

        image = image.to(DEVICE)

        return image, (h, w)

In [48]:
def image2tensor_without_water_net(raw_image, input_size=518):
        transform_resize = Compose([
            Resize(
                width=input_size,
                height=input_size,
                resize_target=False,
                keep_aspect_ratio=True,
                ensure_multiple_of=14,
                resize_method='lower_bound',
                image_interpolation_method=cv2.INTER_CUBIC,
            ),
        ])

        h, w = raw_image.shape[:2]

        image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB) / 255.0
        image = transform_resize({'image': image})['image']


        transform_prepare = Compose([
            NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            PrepareForNet(),
        ])

        image = transform_prepare({'image': image})['image']

        image = torch.from_numpy(image)

        image = image.to(DEVICE)

        return image, (h, w)

In [49]:
class DataFrameDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.features = dataframe.iloc[:, :-1].values  # All columns except last (features)
        self.labels = dataframe.iloc[:, -1].values      # Last column (labels)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_image = cv2.imread(f"data_images/{row['images']}")
        image_tensor, shape = image2tensor(raw_image)

        # Extract target coordinates
        target_coords = [row['pixel_x'], row['pixel_y']]

        # Ground truth depth at target
        gt_depth = row['Act. Range']

        # Reference coordinates
        ref_coords = [row['ref_pixel_x'], row['ref_pixel_y']]

        # Reference distance
        ref_distance = row['ref_distance']

        return image_tensor, shape, target_coords, gt_depth, ref_coords, ref_distance

In [50]:
from torch.utils.data import DataLoader, random_split


def create_dataloaders():
    df = prepare_data()

    dataset = DataFrameDataset(df)

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, shuffle=True)
    test_loader = DataLoader(test_dataset, shuffle=False)

    return train_loader, test_loader

In [51]:
train_loader, test_loader = create_dataloaders()

In [52]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [61]:
def train_model():
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)
    model.to(DEVICE)

    scaler = torch.amp.GradScaler(DEVICE)

    num_epochs = 2

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            gc.collect()
            torch.cuda.empty_cache()
            images, shape, target_coords, gt_depth, ref_coords, ref_distance = batch


            images = images.to(DEVICE)
            if isinstance(target_coords, torch.Tensor):
                target_coords = target_coords.to(DEVICE)
            if isinstance(gt_depth, torch.Tensor):
                gt_depth = gt_depth.to(DEVICE)
            if isinstance(ref_coords, torch.Tensor):
                ref_coords = ref_coords.to(DEVICE)
            if isinstance(ref_distance, torch.Tensor):
                ref_distance = ref_distance.to(DEVICE)

            optimizer.zero_grad()


            with torch.autocast(device_type=DEVICE, dtype=torch.float16):

              predicted_disparity = model(images)

              if predicted_disparity.dim() == 3:
                predicted_disparity = predicted_disparity.unsqueeze(1)

              size = tuple(shape)

              predicted_disparity = F.interpolate(
                  predicted_disparity,
                  size=size,
                  mode="bilinear",
                  align_corners=True,
              )

              predicted_disparity = (predicted_disparity - torch.min(predicted_disparity)) / (torch.max(predicted_disparity) - torch.min(predicted_disparity)) * 255

              predicted_disparity = predicted_disparity.squeeze(1).cpu()
              gt_depth = gt_depth.cpu()
              ref_distance = ref_distance.cpu()


              loss = custom_loss(predicted_disparity, target_coords, gt_depth, ref_coords, ref_distance)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), 'fine_tuned_depthanything_v2.pth')
    print(f"model saved to fine_tuned_depthanything_v2.pth")


In [ ]:
train_model()

In [ ]:
#predict vanilla
model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load('Depth-Anything-V2/checkpoints/depth_anything_v2_vitl.pth', map_location=DEVICE))
model.to(DEVICE)
model.eval()


total_errs = []
long_distance_errs = []
short_distance_errs = []

for batch in test_loader:

    images, shape, target_coords, gt_depth, ref_coords, ref_distance = batch
    images = images.to(DEVICE)

    with torch.no_grad():
        predicted_disparity_eval = model(images)
        if predicted_disparity_eval.dim() == 3:
            predicted_disparity_eval = predicted_disparity_eval.unsqueeze(1)

        size = tuple(shape)

        predicted_disparity_eval = F.interpolate(
            predicted_disparity_eval,
            size=size,
            mode="bilinear",
            align_corners=True,
        )

        eps = 1e-6

        predicted_disparity_eval = predicted_disparity_eval.squeeze(1).cpu()
        gt_depth = gt_depth.cpu()
        ref_distance = ref_distance.cpu()

        disparity_value_ref = predicted_disparity_eval[0][ref_coords[1], ref_coords[0]]
        disparity_value_target = predicted_disparity_eval[0][target_coords[1], target_coords[0]]

        relative_depth_ref = 1.0 / (disparity_value_ref + eps)
        relative_depth_target = 1.0 / (disparity_value_target + eps)

        scale = ref_distance / relative_depth_ref

        predicted_depth = relative_depth_target * scale

        err = abs(predicted_depth - gt_depth) / gt_depth
        total_errs.append(err)

        if gt_depth > 4000:
            long_distance_errs.append(err)
        else:
            short_distance_errs.append(err)



print(f'average error total data: {sum(total_errs).item() / len(total_errs)}')
print(f'standard deviation total data: {math.sqrt(sum([err**2 for err in total_errs]).item()) / len(total_errs)}')
print("*****************************************")
print(f'average error long distance data: {sum(long_distance_errs).item() / len(long_distance_errs)}')
print(f'standard deviation long distance data: {math.sqrt(sum([err**2 for err in long_distance_errs]).item()) / len(long_distance_errs)}')
print("*****************************************")
print(f'average error short distance data: {sum(short_distance_errs).item() / len(short_distance_errs)}')
print(f'standard deviation short distance data: {math.sqrt(sum([err**2 for err in short_distance_errs]).item()) / len(short_distance_errs)}')


In [ ]:
#predict fine tuned
model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load('fine_tuned_depthanything_v2.pth', map_location=DEVICE))
model.to(DEVICE)
model.eval()


total_errs = []
long_distance_errs = []
short_distance_errs = []

for batch in test_loader:

    images, shape, target_coords, gt_depth, ref_coords, ref_distance = batch
    images = images.to(DEVICE)

    with torch.no_grad():
        predicted_disparity_eval = model(images)
        if predicted_disparity_eval.dim() == 3:
            predicted_disparity_eval = predicted_disparity_eval.unsqueeze(1)

        size = tuple(shape)

        predicted_disparity_eval = F.interpolate(
            predicted_disparity_eval,
            size=size,
            mode="bilinear",
            align_corners=True,
        )

        eps = 1e-6

        predicted_disparity_eval = predicted_disparity_eval.squeeze(1).cpu()
        gt_depth = gt_depth.cpu()
        ref_distance = ref_distance.cpu()

        disparity_value_ref = predicted_disparity_eval[0][ref_coords[1], ref_coords[0]]
        disparity_value_target = predicted_disparity_eval[0][target_coords[1], target_coords[0]]

        relative_depth_ref = 1.0 / (disparity_value_ref + eps)
        relative_depth_target = 1.0 / (disparity_value_target + eps)

        scale = ref_distance / relative_depth_ref

        predicted_depth = relative_depth_target * scale

        err = abs(predicted_depth - gt_depth) / gt_depth
        total_errs.append(err)

        if gt_depth > 4000:
            long_distance_errs.append(err)
        else:
            short_distance_errs.append(err)

print(f'average error total data: {sum(total_errs).item() / len(total_errs)}')
print(f'standard deviation total data: {math.sqrt(sum([err**2 for err in total_errs]).item()) / len(total_errs)}')
print("*****************************************")
print(f'average error long distance data: {sum(long_distance_errs).item() / len(long_distance_errs)}')
print(f'standard deviation long distance data: {math.sqrt(sum([err**2 for err in long_distance_errs]).item()) / len(long_distance_errs)}')
print("*****************************************")
print(f'average error short distance data: {sum(short_distance_errs).item() / len(short_distance_errs)}')
print(f'standard deviation short distance data: {math.sqrt(sum([err**2 for err in short_distance_errs]).item()) / len(short_distance_errs)}')
